# PhishGuard: Data Cleaning and Exploratory Analysis

This notebook profiles the source CSV, invokes the reusable cleaning pipeline, checks the cleaned dataset, and reviews class balance and basic URL characteristics. The original CSV in `ml/data/raw/` is treated as immutable. URL-level EDA variables are temporary and are not added to `cleaned_urls.csv`.

In [ ]:
from pathlib import Path
import sys

import pandas as pd

project_root = next(
    candidate
    for candidate in (Path.cwd(), *Path.cwd().parents)
    if (candidate / "ml" / "data" / "raw").is_dir()
)
raw_path = project_root / "ml" / "data" / "raw" / "malicious_phish.csv"
sys.path.insert(0, str(project_root))
from ml.preprocessing.clean_dataset import detect_columns

raw_df = pd.read_csv(raw_path)
print(f"Dataset shape: {raw_df.shape}")
print("First 10 rows:")
display(raw_df.head(10))
print("\nColumn names:", list(raw_df.columns))
print("\nData types:\n", raw_df.dtypes)
print("\nMissing values:\n", raw_df.isna().sum())
print("\nExact duplicate records:", int(raw_df.duplicated().sum()))
print("\nUnique categorical values:")
for column in raw_df.select_dtypes(include=["object", "string", "category"]):
    values = raw_df[column].dropna().unique().tolist()
    print(f"{column}: {values if len(values) <= 30 else f'{len(values)} unique; first 30: {values[:30]}'}")
url_column, label_column = detect_columns(raw_df)
print(f"\nDetected URL column: {url_column}")
print(f"Detected label column: {label_column}")

## Label mapping, cleaning, and validation

Labels are inspected before encoding. The shared preprocessing script detects duplicate/conflicting URLs, separates obviously unprocessable URLs, writes the report and plots, then reloads and validates the cleaned CSV. Conflicting-label URL groups are excluded from training output and retained in the leakage report; unusual but processable URLs are not discarded as outliers.

In [ ]:
from ml.preprocessing.clean_dataset import run_cleaning

results = run_cleaning(raw_path)
cleaned_df = pd.read_csv(results["processed_path"])
print("\nCleaned dataset preview:")
display(cleaned_df.head(10))
print("\nFinal validation checks:")
for check, passed in results["validation"].items():
    print(f"{check}: {'PASS' if passed else 'FAIL'}")
print("\nData dictionary:")
display(pd.read_csv(project_root / "ml" / "data" / "processed" / "data_dictionary.csv"))

## Class balance and URL-level characteristics

The plots below compare legitimate and phishing classes using basic URL metrics only. Outlier values are reported by the cleaning script and retained; these variables are exploratory and do not enter the cleaned training CSV.

In [ ]:
from IPython.display import Image, display

processed_dir = project_root / "ml" / "data" / "processed"
eda_dir = processed_dir / "eda"
class_counts = cleaned_df["label"].value_counts().sort_index()
class_names = {0: "Legitimate", 1: "Phishing", 2: "Defacement", 3: "Malware"}
class_table = pd.DataFrame(
    {
        "class": [class_names[int(code)] for code in class_counts.index],
        "count": class_counts.values,
        "percentage": (class_counts / len(cleaned_df) * 100).round(2).values,
    }
)
print("Class distribution:")
display(class_table)
print("\nURL characteristics by class:")
display(pd.read_csv(eda_dir / "class_url_characteristics.csv"))
print("\nOverall URL feature summary:")
display(pd.read_csv(eda_dir / "url_feature_summary.csv"))
print("\nSaved EDA visualizations:")
for plot_path in sorted(eda_dir.glob("*.png")):
    print(plot_path.name)
    display(Image(filename=str(plot_path)))

### Q&A
- The source dataset contains 1,000 rows and columns `URL` and `Label`. Labels were mapped as `legitimate → 0` and `phishing → 1` after inspecting the source values.
- The cleaned dataset contains 779 rows: 217 duplicate URL rows were removed, and 4 unique URLs were excluded as unprocessable. Six source rows were invalid before duplicate removal; none of the source rows had missing values.

### Data Analysis Key Findings
- The final distribution is 279 legitimate URLs (35.82%) and 500 phishing URLs (64.18%), an imbalance ratio of about 1.79:1. All 217 duplicate URL rows carried the legitimate label, so deduplication changed the class balance.
- No URL had conflicting labels. There are 80 groups of distinct URLs sharing scheme, host, and path when query/fragment are ignored; none cross labels. These remain flagged for grouped splitting in future evaluation.
- Average URL length was 25.32 characters for legitimate URLs and 62.42 for phishing URLs. Average digit counts were 0.00 vs 2.92, and average special-character counts were 5.12 vs 9.40, respectively.
- The six invalid source rows contained unescaped whitespace. No extreme URL-length, digit-count, or special-character outliers exceeded the report thresholds; processable unusual URLs were retained.

### Insights or Next Steps
- `cleaned_urls.csv` passed checks for missing URLs/labels, duplicate URLs, normalized labels, string URLs, reloadability, and presence of both target classes; it is ready for feature engineering.
- For later model evaluation, split by host/path groups to reduce template leakage and address the imbalance using training-only techniques.